## Create and Destroy a Container Registry in Azure using Terraform

- An Azure Container Registry is used to store Images (similar to DockerHub).
  - Since an Azure Container Registry is an Azure resource, it must be placed in an Azure Resource Group.
- This Terraform Project consists of the Terraform files listed below:

In [47]:
!dir *.tf
#!ls *.tf # use this on Linux/Mac

 Volume in drive C is Windows
 Volume Serial Number is 3C6C-8E33

 Directory of c:\Users\PAGA\projects\devops\workshop5\01_Azure_and_Terraform\03_container_registry

01/27/2025  17:55             1,240 container-registry.tf
01/27/2025  17:44               321 providers.tf
01/27/2025  17:44               152 resource-group.tf
01/27/2025  17:52               268 variables.tf
               4 File(s)          1,981 bytes
               0 Dir(s)  41,109,561,344 bytes free


## Terraform providers

- We are using the same Terraform proviers as before (i.e. the `azurerm` provider for Azure).

In [48]:
!type providers.tf
#!cat providers.tf # use this on Linux/Mac

# Initialises Terraform providers and sets their version numbers.

terraform {
  required_providers {
    azurerm = {
      source  = "hashicorp/azurerm"
      version = "~> 4.14.0"
    }
  }

  required_version = ">= 1.10.3"
}

provider "azurerm" {
  features {}
  subscription_id = var.subscription_id
}


## Terraform variables

- We are using the same Terraform variables as before.

**Note! Make sure you change the value for the variable `app_name` to something unique and set your `subscription_id`!**

In [49]:
!type variables.tf
#!cat variables.tf # use this on Linux/Mac

# Sets global variables for this Terraform project.

variable "subscription_id" {
  description = "The Azure subscription ID"
  type        = string
}

variable "app_name" {
  default = "flixtube2025g00"
}

variable "location" {
  default = "westeurope"
}


## Azure Resource Group

- We are using the same Azure Resoure Group as before.

In [50]:
!type resource-group.tf
#!cat resource-group.tf # use this on Linux/Mac

# Creates a resource group in your Azure account.

resource "azurerm_resource_group" "main" {
  name     = var.app_name
  location = var.location
}


## Let's view the contents of the file `container-registry.tf`

- Here we are defining an Azure Resource Group
  - The Block Type is `resource`.
  - The first Block Label is `azurerm_container_registry`
    - `azurerm` is the name of the provider (i.e. the provider for Azure defined in the file `providers.tf`).
    - `container_registry` is the name of the Azure resource (i.e. an Azure Container Registry defined in the `azurerm` provider/plugin).
  - The first Argument sets the Azure Container Registry's name
    - `name` is the argument's name
    - Its value is retrieved from the Terraform variable `app_name` (defined in the file `variables.tf`).
  - The second Argument sets the Azure Resource Group in which the Azure Container Registry will be created
    - `resource_group_name` is the argument's name
    - Its value is retrieved from the Terraform Expression `azurerm_resource_group.main.name`.
      - The `azurerm_resource_group.main` Block is defined in `resource-group.tf` as `resource "azurerm_resource_group" "main"`.
      - In this Block, there is an Argument with a name of `name` who's value is defined as `var.app_name`.
      - This is the value that is assigned to `resource_group_name`.
  - The third Argument sets the Azure Container Registry's Location
    - `location` is the argument's name
    - Its value is retrieved from the Terraform variable `location` (defined in the file `variables.tf`).
  - The fourth Argument enables the Azure Container Registry's Amin Account
    - `admin_enabled` is the argument's name
    - Its value is set to `true`.
    - For more information about the Azure Container Registry's Admin accoun, see:
      - https://learn.microsoft.com/en-us/azure/container-registry/container-registry-authentication?tabs=azure-cli
  - The fifth Argument sets the Azure Container Registry's service tier (also known as SKU).
    - `sku` is the argument's name
    - Its value is set to `"Basic"`.
    - For more information about Azure Container Registry SKUs, see:
      - https://learn.microsoft.com/en-us/azure/container-registry/container-registry-skus.
- The outputs are used to print out the Azure Container Registry's hostname, username and password.
  - The password is sensitive and will be redacted when printed out.
  - All output values can be obtained with the command `terraform output --json` after a `terraform apply`

In [52]:
!type container-registry.tf
#!cat container-registry.tf # use this on Linux/Mac

# Creates a container registry in Azure (for Docker images).
# Note!
# - Resource "azurerm_resource_group.main" with a property "name" is defined in the file "resource-group.tf".
# - The value for "resource_group_name" below is set using property "name" in resource "azurerm_resource_group.main":
#   - resource_group_name = azurerm_resource_group.main.name
# - "name" and "location" below are set from Terraform variables defined in the file "variables.tf".
# - The ouputs below are used to print out the Azure Container Registry's hostname, username and password.
#   - The password is sensitive and will be redacted when printed out.


resource "azurerm_container_registry" "main" {
  name                = var.app_name
  resource_group_name = azurerm_resource_group.main.name
  location            = var.location
  admin_enabled       = true
  sku                 = "Basic"
}

output "AZURE_CONTAINER_REGISTRY_HOSTNAME" {
  value = azurerm_container_registry.main.login_server
}

output "AZURE_CO

## Initialize Terraform

- The Terraform CLI command `terraform init` initializes the Terraform Project.

In [53]:
#rm -rf .terraform rm .terraform.lock.hcl terraform.tfstate terraform.tfstate.backup
!terraform init

Initializing the backend...
Initializing provider plugins...
- Finding hashicorp/azurerm versions matching "~> 4.14.0"...
- Installing hashicorp/azurerm v4.14.0...
- Installed hashicorp/azurerm v4.14.0 (signed by HashiCorp)
Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.


## Terraform Apply

- The Terraform CLI command `terraform apply -auto-approve` applies (creates/updates) the resources defined in the Terraform project.

**Note!**

- Once again it's better to run this command in a separate terminal.
  - Open a new terminal.
  - Make sure you are in the folder `workshop5/01_Azure_and_Terraform/03_container_registry`
  - Execute the command `terraform apply -auto-approve`

The output should look something like the below ...

```bash
Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # azurerm_container_registry.main will be created
  + resource "azurerm_container_registry" "main" {
      + admin_enabled                 = true
      + admin_password                = (sensitive value)
      + admin_username                = (known after apply)
      + encryption                    = (known after apply)
      + export_policy_enabled         = true
      + id                            = (known after apply)
      + location                      = "westeurope"
      + login_server                  = (known after apply)
      + name                          = "flixtube2025g00"
      + network_rule_bypass_option    = "AzureServices"
      + network_rule_set              = (known after apply)
      + public_network_access_enabled = true
      + resource_group_name           = "flixtube2025g00"
      + sku                           = "Basic"
      + trust_policy_enabled          = false
      + zone_redundancy_enabled       = false
    }

  # azurerm_resource_group.main will be created
  + resource "azurerm_resource_group" "main" {
      + id       = (known after apply)
      + location = "westeurope"
      + name     = "flixtube2025g00"
    }

Plan: 2 to add, 0 to change, 0 to destroy.

Changes to Outputs:
  + AZURE_CONTAINER_REGISTRY_HOSTNAME = (known after apply)
  + AZURE_CONTAINER_REGISTRY_PASSWORD = (sensitive value)
  + AZURE_CONTAINER_REGISTRY_USERNAME = (known after apply)
azurerm_resource_group.main: Creating...
azurerm_resource_group.main: Still creating... [10s elapsed]
azurerm_resource_group.main: Creation complete after 12s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_container_registry.main: Creating...
azurerm_container_registry.main: Still creating... [10s elapsed]
azurerm_container_registry.main: Still creating... [20s elapsed]
azurerm_container_registry.main: Creation complete after 28s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]

Apply complete! Resources: 2 added, 0 changed, 0 destroyed.

Outputs:

AZURE_CONTAINER_REGISTRY_HOSTNAME = "flixtube2025g00.azurecr.io"
AZURE_CONTAINER_REGISTRY_PASSWORD = <sensitive>
AZURE_CONTAINER_REGISTRY_USERNAME = "flixtube2025g00"
```

In [ ]:
# !terraform apply -auto-approve

## Get the Values of Sensitive Outputs

- Run the cell below.
- Notice that the output for `AZURE_CONTAINER_REGISTRY_PASSWORD` has been redacted by Terraform since its a sensitive value.
- To get the value of all outputs, we can use the command `terraform output --json`.
- Notice that the sensitive output variable's value is shown in the JSON document below.

In [ ]:
!terraform output --json

## List Azure Resource Groups

- The Azure CLI command `az group list -o table` lists all Resource Groups in Azure.
- We see that the Resource Group defined in the Terraform project has been created.

You can also visit https://portal.azure.com/#browse/resourcegroups to see the resource group.

In [55]:
!az group list -o table

Name             Location    Status
---------------  ----------  ---------
flixtube2025g00  westeurope  Succeeded


## List Resources in Resource Group `flixtube2025g00`

- The Azure CLI command `az resource list -n flixtube2025g00 -o table` lists resources in Resource Group `flixtube2025g00`.
- We see that the Resource Group contains the Container Registry.

You can also visit https://portal.azure.com/#browse/all to see all resources, where the Container registry is listed.

In [56]:
!az resource list -n flixtube2025g00 -o table

Name             ResourceGroup    Location    Type                                    Status
---------------  ---------------  ----------  --------------------------------------  --------
flixtube2025g00  flixtube2025g00  westeurope  Microsoft.ContainerRegistry/registries


## List Azure Container Registries

- The Azure CLI command `az acr list -o table` lists all Container Registries in Azure.
- We see that the Container Registry defined in the Terraform project has been created.

In [57]:
!az acr list -o table

NAME             RESOURCE GROUP    LOCATION    SKU    LOGIN SERVER                CREATION DATE         ADMIN ENABLED
---------------  ----------------  ----------  -----  --------------------------  --------------------  ---------------
flixtube2025g00  flixtube2025g00   westeurope  Basic  flixtube2025g00.azurecr.io  2025-01-28T04:45:57Z  True


## List Repositories in Container Registry `flixtube2025g00 `

- The Azure CLI command `az acr repository list -n flixtube2025g00 --top 10 -o table` lists Repositories in a Azure Container Registry `flixtube2025g00`.
  - The `-n` option is manditory and specifies the `NAME` of the Container Registry.
  - The `--top 10` limits the list to the first 10 Repositories (remove to see all Repositories).
- We see that Container Registry `flixtube2025g00` doesn't contain any Repositories.

In [58]:
!az acr repository list -n flixtube2025g00 --top 10 -o table

## Show Information about Azure Container Registry `flixtube2025g00`

- The Azure CLI command `az acr show -n flixtube2025g00 -o table` shows information about Container Registry `flixtube2025g00`.
- It shows the Container Registry's `LOGIN SERVER` which is the URL to your Container Registry on Azure.
  - **This is the URL you would use to upload Docker Images to the Azure Container Registry.**

In [ ]:
!az acr show -n flixtube2025g00 -o table

# Let's store the LOGIN SERVER in a Python variable so we can use it later in this notebook
CONTAINER_REGISTRY_LOGIN_SERVER=!az acr show -n flixtube2025g00 --query loginServer -o tsv
CONTAINER_REGISTRY_LOGIN_SERVER=CONTAINER_REGISTRY_LOGIN_SERVER[0]

## Show Credentials for Azure Container Registry `flixtube2025g00`

- The Azure CLI command `az acr credential show -n flixtube2025g00 -o table` shows credentials about Container Registry `flixtube2025g00`.
- It shows the Container Registry's `USERNAME` and  `PASSWORD` to use to authenticate with your Azure Container Registry.
  - **This is the USERNAME and PASSWORD you would use to login to Docker to upload Images to the Azure Container Registry.**

In [ ]:
!az acr credential show -n flixtube2025g00 -o table

# Let's store the USERNAME and PASSWORD in Python variables so we can use them later in this notebook
CONTAINER_REGISTRY_USERNAME=!az acr credential show -n flixtube2025g00 --query username -o tsv
CONTAINER_REGISTRY_USERNAME=CONTAINER_REGISTRY_USERNAME[0]
CONTAINER_REGISTRY_PASSWORD=!az acr credential show -n flixtube2025g00 --query passwords[0].value -o tsv
CONTAINER_REGISTRY_PASSWORD=CONTAINER_REGISTRY_PASSWORD[0]

## Login to Azure Container Registry via Docker

- Replace the variables below with your `LOGIN_SERVER`, `USERNAME` and `PASSWORD`.

In [61]:
!docker login $CONTAINER_REGISTRY_LOGIN_SERVER -u $CONTAINER_REGISTRY_USERNAME -p $CONTAINER_REGISTRY_PASSWORD

# In Ubuntu with environment variables CONTAINER_REGISTRY_LOGIN_SERVER, CONTAINER_REGISTRY_USERNAME and CONTAINER_REGISTRY_USERNAME
#!echo $PASSWORD | docker login $LOGIN_SERVER -u $USERNAME --password-stdin  > /dev/null 2>&1

Login Succeeded


WARNING! Using --password via the CLI is insecure. Use --password-stdin.


## Let's view the code in `Flixtube.VideoStreaming`

- The file `Flixtube.VideoStreaming/Flixtube.VideoStreaming/Program.cs` is listed below.
  - At the top of the file, it reads in a couple of environment variables.
    - `FLIXTUBE_VIDEO_STREAMING_PORT` is the port the microservice will listen on.
    - `FLIXTUBE_STORAGE_FOLDER_NAME` is the name of the filesystem folder it will serve video files from.
  - These environment variables are stored in the `IConfiguration` instance, after discarding their `FLIXTUBE_` prefix.
  - Then it configures the service container and the HTTP Request/Response pipeline as usual for an ASP.NET Web API project.
  - Lastly, it starts the microservice listening on port `FLIXTUBE_VIDEO_STREAMING_PORT`.

In [63]:
!type Flixtube.VideoStreaming\Flixtube.VideoStreaming\Program.cs
#!cat Flixtube.VideoStreaming/Flixtube.VideoStreaming/Program.cs # use this on Linux/Mac

using Scalar.AspNetCore;

var builder = WebApplication.CreateBuilder(args);

// Make sure the necessary environment variables are available.

if (string.IsNullOrEmpty(Environment.GetEnvironmentVariable("FLIXTUBE_VIDEO_STREAMING_PORT"))) {
    throw new Exception("Please specify the port number for Flixtube.VideoStreaming with the environment variable FLIXTUBE_VIDEO_STREAMING_PORT.");
}

if (string.IsNullOrEmpty(Environment.GetEnvironmentVariable("FLIXTUBE_STORAGE_FOLDER_NAME"))) {
    throw new Exception("Please specify the Filesystem folder name for Flixtube.VideoStreaming with the subkey FLIXTUBE_STORAGE_FOLDER_NAME.");
}

// Get necessary environment variables
// Note that we only need to get settings here if there are need before builder.Build()
// int VIDEO_STREAMING_PORT = int.Parse(Environment.GetEnvironmentVariable("FLIXTUBE_VIDEO_STREAMING_PORT") ?? "80");
// string STORAGE_FOLDER_NAME = Environment.GetEnvironmentVariable("FLIXTUBE_STORAGE_FOLDER_NAME") ?? string.Empty;

// On

- The file `Flixtube.VideoStreaming/Flixtube.VideoStreaming/Controllers/VideoStreamingController.cs` is listed below.
  - At the top of the file, it reads in an environment variable stored in the dependency-injected `IConfiguration` instance.
    - `STORAGE_FOLDER_NAME` is the name of the filesystem folder the microservice will serve video files from.
  - The file also contains a number of HTTP endpoints, where one of these is the `/{id}` route that streams a video.
    - `{id}` is the name of the video file in the microservice's filesystem.
    - `STORAGE_FOLDER_NAME` is the name of the folder the microservice serves video files from.
    - `STORAGE_FOLDER_NAME/id` is the path to the video file in the microservice's filesystem.
- So the microservice will simply stream video files stored in its filesystem when `HTTP GET /{id}` is called. 

In [64]:
!type Flixtube.VideoStreaming\Flixtube.VideoStreaming\Controllers\VideoStreamingController.cs
#!cat Flixtube.VideoStreaming/Flixtube.VideoStreaming/Controllers/VideoStreamingController.cs # use this on Linux/Mac

using System.Net;
using Microsoft.AspNetCore.Mvc;

namespace Flixtube.VideoStreaming.Controllers;

[ApiController]
[Route("/")]
public class VideoStreamingController : ControllerBase
{
    private readonly ILogger<VideoStreamingController> _logger;
    private readonly IConfiguration _config;
    private readonly string STORAGE_FOLDER_NAME;

    public VideoStreamingController(ILogger<VideoStreamingController> logger, IConfiguration config)
    {
        _logger = logger;
        _config = config;

        STORAGE_FOLDER_NAME = _config.GetValue<string>("STORAGE_FOLDER_NAME")!;

        _logger.LogInformation("VideoStreamingController() called.");
    }

    // Health check.
    [HttpGet("/health")]
    public async Task<IActionResult> Health()
    {
        await Task.Delay(0);
        return Ok();
    }

    // Stream video from the Filesystem.
    [HttpGet("{id}")]
    public async Task StreamVideo(string id)
    {
        // _logger.LogInformation($"StreamVideo() called.");
        

## Let's look at the Docker file for `Flixtube.VideoStreaming`

- The Dockerfile `Flixtube.VideoStreaming\Dockerfile` is listed below.
  - It base image has `.net sdk 9.0` pre-installed.
  - It sets the `WORKDIR` to `/src` and copies all code for the microservice to it.
  - It also copies the `videos` folder to `/src`.
    - This folder contains one sample video file `SampleVideo_1280x720_1mb-mp4`.
  - Then it restores (installs) all NuGet packages.
  - Finally, it starts the microservice, by issuing the command below when the container starts.
    - `dotnet watch run --no-launch-profile --project Flixtube.VideoStreaming.csproj`

In [65]:
!type Flixtube.VideoStreaming\Dockerfile
#!cat Flixtube.VideoStreaming/Dockerfile # use this on Linux/Mac

FROM mcr.microsoft.com/dotnet/sdk:9.0
WORKDIR /src
EXPOSE 80
COPY ./Flixtube.VideoStreaming ./
COPY ./videos ./videos
RUN ["dotnet","restore"]
CMD ["dotnet","watch","run","--no-launch-profile","--project","Flixtube.VideoStreaming.csproj"]


## Build and Push a Docker Image to Azure Container Registry

- Here we are building an image of the ASP.NET Web API application (microservice).
- We are tagging the image as `flixtube2025g00.azurecr.io/video-streaming:1,`where:
  - `flixtube2025g00.azurecr.io` is the URL (LOGIN SERVER) to our Container Registry.
  - `video-streaming` is the name of our image (repository).
  - `1` is the version of the image (tag).
- Then the image is pushed to the Azure Container Registry.
- Finally, the local image is removed from the host computer.

In [66]:
# Build Docker image with Nodejs Application
!docker build -q -t {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1 -f ./Flixtube.VideoStreaming/Dockerfile ./Flixtube.VideoStreaming

# Push Docker Image to Azure Container Registry
!docker push {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1

# Clean up
!docker rmi {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1
!docker images {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1

sha256:4027094e62881b5332b674e1b6b47f1d9f5545d78812634a3a63bf2c32bd3e7f
The push refers to repository [flixtube2025g00.azurecr.io/video-streaming]
45faf544d0e6: Preparing
dcffe1b8a5c5: Preparing
515210e4a4a7: Preparing
cb5d5d0673d7: Preparing
1bb564ecf252: Preparing
7e9d6a3f8c92: Preparing
2e7a3a1e4448: Preparing
064dc71a1978: Preparing
c9aaa778af8e: Preparing
5479f1788e98: Preparing
d54764d9a8e7: Preparing
3b245d6409b1: Preparing
f5fe472da253: Preparing
064dc71a1978: Waiting
c9aaa778af8e: Waiting
5479f1788e98: Waiting
d54764d9a8e7: Waiting
3b245d6409b1: Waiting
f5fe472da253: Waiting
7e9d6a3f8c92: Waiting
2e7a3a1e4448: Waiting
515210e4a4a7: Pushed
dcffe1b8a5c5: Pushed
cb5d5d0673d7: Pushed
1bb564ecf252: Pushed
2e7a3a1e4448: Pushed
c9aaa778af8e: Pushed
45faf544d0e6: Pushed
064dc71a1978: Pushed
d54764d9a8e7: Pushed
3b245d6409b1: Pushed
5479f1788e98: Pushed
f5fe472da253: Pushed
7e9d6a3f8c92: Pushed
1: digest: sha256:95e0a8492c75f330ee827f0e798df1e9c9962a1bce58cf4b4d88b90586325387 size: 305

## List Repositories in Container Registry `flixtube2025g00 `

- We see that the Repository `video-streaming` has been created in Container Registry `flixtube2025g00`.

You can also see the Container Registry on Azure.
- Visit https://portal.azure.com/#browse/all
- Click the Container Registry `flixtube2025g00`
- Then expand `Services` and choose `Repositories` to see the repository `video-streaming`.
- If you click repository `video-streaming`, you will see that it has one tag `1`.

In [67]:
!az acr repository list -n flixtube2025g00 --top 10 -o table

Result
---------------
video-streaming


## Show Information about Repository `video-streaming`

- The Azure CLI command `az acr repository show -n flixtube2025g00 --repository video-streaming -o table`
  - Shows information about Repository `video-streaming` in Container Registry `flixtube2025g00`.
    - It contains images named `video-streaming` (ImageName).
    - It has a tag count of `1` (TagCount), i.e. currently there is only one tag for the `video-streaming` image.

In [68]:
!az acr repository show -n flixtube2025g00 --repository video-streaming -o table

CreatedTime                   ImageName        LastUpdateTime                ManifestCount    Registry                    TagCount
----------------------------  ---------------  ----------------------------  ---------------  --------------------------  ----------
2025-01-28T05:15:10.3559425Z  video-streaming  2025-01-28T05:15:10.4471221Z  1                flixtube2025g00.azurecr.io  1


## List Tags in Repository `video-streaming`

- The Azure CLI command `az acr repository show-tags -n flixtube2025g00 --repository video-streaming --top 10 -o table`:
  - Lists the tags in Repository `video-streaming` in Container Registry `flixtube2025g00`.
    - Currently there is only one tag.
    - The tag has the value `1`.

In [69]:
!az acr repository show-tags -n flixtube2025g00 --repository video-streaming --top 10 -o table

Result
--------
1


## Show Information about Image `video-streaming:1`

- The Azure CLI command `az acr repository show -n flixtube2025g00 --image video-streaming:1 -o table`:
  - Shows information about image `video-streaming:1` in Container Registry `flixtube2025g00`.
    - The information includes the Digest for the image.

In [70]:
!az acr repository show -n flixtube2025g00 --image video-streaming:1 -o table

CreatedTime                   Digest                                                                   LastUpdateTime                Name    Signed
----------------------------  -----------------------------------------------------------------------  ----------------------------  ------  --------
2025-01-28T05:15:10.4743775Z  sha256:95e0a8492c75f330ee827f0e798df1e9c9962a1bce58cf4b4d88b90586325387  2025-01-28T05:15:10.4743775Z  1       False


## Show Azure Container Registry Usage

- The command `az acr show-usage -n flixtube2025g00 -o table` shows the usage of Container Registry `flixtube2025g00`.
  - We can see the maximum number of allowed bytes in the `LIMIT` column (first row).
  - We can see the current number of bytes in the `CURRENT VALUE` column (first row).

In [71]:
!az acr show-usage -n flixtube2025g00 -o table

NAME       LIMIT        CURRENT VALUE    UNIT
---------  -----------  ---------------  ------
Size       10737418240  318026174        Bytes
Webhooks   2            0                Count
ScopeMaps  100          0                Count
Tokens     100          0                Count


## Pull and Run an Image from the Azure Container Registry

- The image `video-streaming:1` will be pulled from the Azure Container Registry to the host computer.
- Then a Container is created from the Image, where:
  - The host computer's port 3000 is mapped to the container's port 3000.
  - An environment variable `FLIXTUBE_VIDEO_STREAMING_PORT` is created in the container with the value `3000`.
  - An environment variable `FLIXTUBE_STORAGE_FOLDER_NAME` is created in the container with the value `videos`.

In [ ]:
!docker run --name video-streaming -d -p 3000:3000 -e FLIXTUBE_VIDEO_STREAMING_PORT=3000 -e FLIXTUBE_STORAGE_FOLDER_NAME=videos {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1
!docker ps
!docker logs video-streaming
#!docker exec -it video-streaming bash

2df8a70127a4e9fed2fb5da0bc0f508669e6c145367b72443357f707f3d22b6a


Unable to find image 'flixtube2025g00.azurecr.io/video-streaming:1' locally
1: Pulling from video-streaming
486dbf987c66: Already exists
b7f90be4bd50: Already exists
56676eeaaf44: Already exists
b7db1e46071c: Already exists
baddccf2b575: Already exists
3820e4df2c07: Already exists
2da257c5207e: Already exists
e5b6c62eef46: Already exists
44fcc8e6ed98: Already exists
750df5a7f22a: Already exists
43eeddf41c34: Already exists
ace035496e0e: Already exists
7e6cca92d493: Already exists
Digest: sha256:95e0a8492c75f330ee827f0e798df1e9c9962a1bce58cf4b4d88b90586325387
Status: Downloaded newer image for flixtube2025g00.azurecr.io/video-streaming:1


CONTAINER ID   IMAGE                                          COMMAND                  CREATED        STATUS                  PORTS                            NAMES
2df8a70127a4   flixtube2025g00.azurecr.io/video-streaming:1   "dotnet watch run --…"   1 second ago   Up Less than a second   80/tcp, 0.0.0.0:3000->3000/tcp   video-streaming
0fcaba79546f   4a7ae7008ea2                                   "/docker-entrypoint.…"   19 hours ago   Up 19 hours                                              k8s_proxy_kubernetes-dashboard-kong-78fd98d579-g489v_kubernetes-dashboard_d9874745-7e26-4ef6-8eea-1f61c8538df6_0
9fe370cc4473   d9cbc9f4053c                                   "/dashboard-metrics-…"   19 hours ago   Up 19 hours                                              k8s_kubernetes-dashboard-metrics-scraper_kubernetes-dashboard-metrics-scraper-6c8d6bb74d-tv5v5_kubernetes-dashboard_5ed529a7-fa15-47d5-98eb-cbc576ff14fc_0
c3b9888b8dd8   538c5083d89d                                   "/dashboard-

## Access the Microservice from a Web Browser

- Open a web browser and enter the URL http://localhost:3000/SampleVideo_1280x720_1mb.mp4
  - This will send an HTTP GET request to the microservice's GET route for the `/{id}` path.
  - The video stored in the microservice's container will be streamed to the web browser.

In [26]:
#!firefox http://localhost:3000/SampleVideo_1280x720_1mb.mp4

## Stop and Remove the Docker Container and Image from your computer

- The Container `video-streaming` is stopped and removed from the host computer.
- Then the Image `video-streaming:1` is removed from the host computer.

In [73]:
!docker stop video-streaming
!docker rm video-streaming
!docker ps -a
!docker rmi {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1
!docker images {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1

video-streaming
video-streaming
CONTAINER ID   IMAGE          COMMAND                  CREATED        STATUS        PORTS     NAMES
0fcaba79546f   4a7ae7008ea2   "/docker-entrypoint.…"   19 hours ago   Up 19 hours             k8s_proxy_kubernetes-dashboard-kong-78fd98d579-g489v_kubernetes-dashboard_d9874745-7e26-4ef6-8eea-1f61c8538df6_0
9fe370cc4473   d9cbc9f4053c   "/dashboard-metrics-…"   19 hours ago   Up 19 hours             k8s_kubernetes-dashboard-metrics-scraper_kubernetes-dashboard-metrics-scraper-6c8d6bb74d-tv5v5_kubernetes-dashboard_5ed529a7-fa15-47d5-98eb-cbc576ff14fc_0
c3b9888b8dd8   538c5083d89d   "/dashboard-auth"        19 hours ago   Up 19 hours             k8s_kubernetes-dashboard-auth_kubernetes-dashboard-auth-f7d869bcb-66r6j_kubernetes-dashboard_aef1e7ca-0388-407b-83f1-16196a54642f_0
834831d96cf5   71e2af47d086   "/dashboard-api --in…"   19 hours ago   Up 19 hours             k8s_kubernetes-dashboard-api_kubernetes-dashboard-api-c9c479bb4-w56zq_kubernetes-dashboard_6

## Logout from the Azure Container Registry via Docker

In [74]:
!docker logout $CONTAINER_REGISTRY_LOGIN_SERVER

Removing login credentials for flixtube2025g00.azurecr.io


## Delete Repository `video-streaming` from Azure Container Registry

- The Azure CLI command `az acr repository delete -y -n flixtube2025g00 --repository video-streaming -o table`
  - Deletes the Repository `video-streaming` in Container Registry `flixtube2025g00`.
  - Use this command to delete all images in Repository `video-streaming`.
- The Azure CLI command `az acr repository delete -y -n flixtube2025g00 --image video-streaming:1 -table`
  - Deletes the Image `video-streaming:1` in Container Registry `flixtube2025g00`.
  - Use this command to delete one image from Repository `video-streaming`.

In [75]:
#!az acr repository delete -y -n flixtube2025g00 --image video-streaming:1 -table
!az acr repository delete -y -n flixtube2025g00 --repository video-streaming -o table

## List Repositories in Container Registry `flixtube2025g00 `

- We see that the Repository `video-streaming` has been deleted from Container Registry `flixtube2025g00`.

In [76]:
!az acr repository list -n flixtube2025g00 --top 10 -o table

## Terraform Destroy

- The Terraform CLI command `terraform destroy -auto-approve` destroys the resources defined in the Terraform project.
**Note!**

- Once again it's better to run this command in a separate terminal.
  - Open a new terminal.
  - Make sure you are in the folder `workshop5/01_Azure_and_Terraform/03_container_registry`
  - Execute the command `terraform destroy -auto-approve`

The output should look something like the below ...

```bash
azurerm_resource_group.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_container_registry.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]

Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # azurerm_container_registry.main will be destroyed
  - resource "azurerm_container_registry" "main" {
      - admin_enabled                 = true -> null
      - admin_password                = (sensitive value) -> null
      - admin_username                = "flixtube2025g00" -> null
      - anonymous_pull_enabled        = false -> null
      - data_endpoint_enabled         = false -> null
      - encryption                    = [] -> null
      - export_policy_enabled         = true -> null
      - id                            = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00" -> null
      - location                      = "westeurope" -> null
      - login_server                  = "flixtube2025g00.azurecr.io" -> null
      - name                          = "flixtube2025g00" -> null
      - network_rule_bypass_option    = "AzureServices" -> null
      - network_rule_set              = [] -> null
      - public_network_access_enabled = true -> null
      - quarantine_policy_enabled     = false -> null
      - resource_group_name           = "flixtube2025g00" -> null
      - retention_policy_in_days      = 0 -> null
      - sku                           = "Basic" -> null
      - tags                          = {} -> null
      - trust_policy_enabled          = false -> null
      - zone_redundancy_enabled       = false -> null
    }

  # azurerm_resource_group.main will be destroyed
  - resource "azurerm_resource_group" "main" {
      - id         = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00" -> null
      - location   = "westeurope" -> null
      - name       = "flixtube2025g00" -> null
      - tags       = {} -> null
        # (1 unchanged attribute hidden)
    }

Plan: 0 to add, 0 to change, 2 to destroy.

Changes to Outputs:
  - AZURE_CONTAINER_REGISTRY_HOSTNAME = "flixtube2025g00.azurecr.io" -> null
  - AZURE_CONTAINER_REGISTRY_PASSWORD = (sensitive value) -> null
  - AZURE_CONTAINER_REGISTRY_USERNAME = "flixtube2025g00" -> null
azurerm_container_registry.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_container_registry.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...nerRegistry/registries/flixtube2025g00, 10s elapsed]
azurerm_container_registry.main: Destruction complete after 16s
azurerm_resource_group.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00, 10s elapsed]
azurerm_resource_group.main: Destruction complete after 19s

Destroy complete! Resources: 2 destroyed.
```

In [40]:
# !terraform destroy -auto-approve

## List Azure Container Registries

- The Azure CLI command `az acr list -o table` lists all Container Registries in Azure.
- We see that the Container Registry defined in the Terraform project has been destroyed.

You can also visit https://portal.azure.com/#browse/all to see all resources, where the Container registry is gone.

In [77]:
!az acr list -o table

## List Azure Resource Groups

- The Azure CLI command `az group list -o table` lists all Resource Groups in Azure.
- We see that the Resource Group defined in the Terraform project has been destroyed.

You can also visit https://portal.azure.com/#browse/resourcegroups to see that the resource group is gone.

In [78]:
!az group list -o table